# 🧹 AgentCore End-to-End 리소스 정리

이 Notebook은 AgentCore End-to-End 튜토리얼에서 생성한 모든 리소스를 종합적으로 정리하는 절차를 제공합니다.

## 개요

이 정리 절차에서는 다음 항목을 제거합니다.
- **Memory**: AgentCore Memory 리소스 및 저장된 데이터
- **Runtime**: 에이전트 런타임 인스턴스 및 ECR 리포지토리
- **보안**: Execution role 및 Authorization Provider 리소스
- **관찰 기능**: CloudWatch log group 및 stream
- **로컬 파일**: 생성된 구성 및 코드 파일

⚠️ **중요**: 정리 작업은 되돌릴 수 없습니다. 계속하기 전에 필요한 중요 데이터를 모두 저장했는지 확인하세요.

---

## 단계 1: 필수 종속성 가져오기

정리 절차에 필요한 모든 모듈과 헬퍼 함수를 불러옵니다.

In [ ]:
# 참고: 자율 학습 실습에서만 주석을 해제하고 실행합니다.
# !aws sts get-caller-identity

# 필수 패키지 설치
# %pip install -r requirements.txt -q

In [ ]:
import json

from lab_helpers.lab2_memory import REGION
from lab_helpers.utils import (
    delete_agentcore_runtime_execution_role,
    delete_ssm_parameter,
    cleanup_cognito_resources,
    get_customer_support_secret,
    delete_customer_support_secret,
    agentcore_memory_cleanup,
    gateway_target_cleanup,
    runtime_resource_cleanup,
    delete_observability_resources,
    local_file_cleanup,
    get_ssm_parameter,
)

print("✅ Dependencies imported successfully")
print(f"🌍 Working in region: {REGION}")

## 단계 2: Memory 리소스 정리

AgentCore Memory 리소스 및 관련 데이터를 제거합니다.

In [ ]:
print("🧠 Starting Memory cleanup...")
agentcore_memory_cleanup(get_ssm_parameter("/app/customersupport/agentcore/memory_id"))

## 단계 3: Runtime 리소스 정리

AgentCore Runtime, ECR 리포지토리 및 관련 AWS 리소스를 제거합니다.

In [ ]:
print("🚀 Starting Runtime cleanup...")
runtime_resource_cleanup(get_ssm_parameter("/app/customersupport/agentcore/runtime_arn"))

## 단계 4: Gateway 리소스 정리
Target과 Gateway를 제거합니다.

In [ ]:
# 선택 사항
# print("⚙️ Starting Policy Engine Cleanup...")
# policy_engine_cleanup(get_ssm_parameter("/app/customersupport/agentcore/policy_engine_id"))

print("⚙️ Starting Gateway Cleanup...")
gateway_target_cleanup(get_ssm_parameter("/app/customersupport/agentcore/gateway_id"))

## 단계 5: 보안 리소스 정리

Execution role 및 인증 리소스를 제거합니다.

In [ ]:
print("🛡️  Starting Security cleanup...")
try:
    # bedrock_client = boto3.client("bedrock", region_name=REGION)

    # execution role 삭제
    print("  🗑️  Deleting AgentCore Runtime execution role...")
    delete_agentcore_runtime_execution_role()
    print("  ✅ Execution role deleted")

    # SSM 파라미터 삭제
    print("  🗑️  Deleting SSM parameter...")
    delete_ssm_parameter("/app/customersupport/agentcore/runtime_arn")
    print("  ✅ SSM parameter deleted")

    # Cognito 및 secret 정리
    print("  🗑️  Cleaning up Cognito resources...")
    cs = json.loads(get_customer_support_secret())
    cleanup_cognito_resources(cs["pool_id"])
    print("  ✅ Cognito resources cleaned up")

    print("  🗑️  Deleting customer support secret...")
    delete_customer_support_secret()
    print("  ✅ Customer support secret deleted")

except Exception as e:
    print(f"  ⚠️  Error during security cleanup: {e}")

## 단계 6: 로컬 파일 정리

로컬 디렉터리에서 생성된 구성 및 코드 파일을 제거합니다.

In [ ]:
print("📁 Starting Local Files cleanup...")
local_file_cleanup()

## 단계 7: 관찰 리소스 정리

에이전트 모니터링에 사용된 CloudWatch 로그 그룹 및 스트림을 제거합니다.

In [ ]:
print("📊 Starting Observability cleanup...")

delete_observability_resources()

## 🎉 정리 완료!

모든 AgentCore 리소스를 정리했습니다. 제거된 항목의 요약은 다음과 같습니다.

In [ ]:
print("\n" + "=" * 60)
print("🧹 CLEANUP COMPLETED SUCCESSFULLY! 🧹")
print("=" * 60)
print()
print("📋 Resources cleaned up:")
print("  🧠 Memory: AgentCore Memory resources and data")
print("  🚀 Runtime: Agent runtime and ECR repository")
print("  🛡️ Security: Roles, and SSM secrets")
print("  📊 Observability: CloudWatch logs")
print("  📁 Files: Local configuration files")
print()
print("✨ Your AWS account is now clean and ready for new experiments!")
print("\nThank you for completing the AgentCore End-to-End tutorial! 🚀")